<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 03 - Decision Tree

En este notebook se entrena un modelo base de **árbol de decisión** para clasificar usuarios de Twitch según la variable `mature`.

El objetivo es comprobar el comportamiento inicial del modelo usando diferentes conjuntos de características:

- solo características relacionales del grafo;
- solo características no relacionales;
- combinación de ambas.

Al final se guarda el modelo base, las métricas obtenidas y la matriz de confusión.

## Objetivo metodologico

Este cuaderno usa Decision Tree como primer clasificador interpretable. Ademas de entrenar el modelo base, compara tres conjuntos de variables: solo relacionales, solo no relacionales y mixtas. Esta comparacion responde directamente al requisito de analizar combinaciones de propiedades relacionales y no relacionales.


## 1. Importación de librerías y configuración inicial

Se importan las funciones reutilizables de `src/`, se definen las rutas principales del proyecto y se prepara el entorno de trabajo.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, get_classification_report, save_metrics, save_model
from src.model_training import get_base_models, split_data
from src.visualization import save_confusion_matrix_plot

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
IMG_DIR = ROOT / "img"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga del dataset procesado y entrenamiento del modelo

Se carga `twitch_mature_features.csv`, se separa la variable objetivo `mature` y se comparan distintos conjuntos de características para seleccionar el más adecuado para Decision Tree.

### Diseno experimental

Se usa una particion estratificada para mantener la proporcion de clases `mature` y `no mature` en entrenamiento y test. El criterio principal es `f1`, porque combina precision y recall y es mas informativo que accuracy cuando las clases no estan perfectamente balanceadas.


In [ ]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
y = df["mature"]

relational_cols = ["degree", "degree_centrality", "clustering", "pagerank", "closeness", "betweenness", "community"]
non_relational_cols = ["num_features", "days", "views"]
feature_sets = {
    "relacional": relational_cols,
    "no_relacional": non_relational_cols,
    "mixto": relational_cols + non_relational_cols,
}

rows = []
for feature_set_name, feature_cols in feature_sets.items():
    X_set = df[feature_cols]
    X_train_set, X_test_set, y_train_set, y_test_set = split_data(X_set, y, test_size=0.2, random_state=42)
    candidate = get_base_models(random_state=42)["decision_tree"]
    candidate.fit(X_train_set, y_train_set)
    train_pred = candidate.predict(X_train_set)
    test_pred = candidate.predict(X_test_set)
    train_metrics = evaluate_model(y_train_set, train_pred)
    test_metrics = evaluate_model(y_test_set, test_pred)
    rows.append({
        "model": "decision_tree",
        "feature_set": feature_set_name,
        **test_metrics,
        "train_f1": train_metrics["f1"],
        "f1_gap": train_metrics["f1"] - test_metrics["f1"],
    })

results_feature_sets = pd.DataFrame(rows).sort_values(["f1", "f1_gap"], ascending=[False, True]).reset_index(drop=True)
best_feature_set = results_feature_sets.iloc[0]["feature_set"]
best_cols = feature_sets[best_feature_set]

X_best = df[best_cols]
X_train, X_test, y_train, y_test = split_data(X_best, y, test_size=0.2, random_state=42)
model = get_base_models(random_state=42)["decision_tree"]
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = evaluate_model(y_test, y_pred)
metrics.update({
    "model": "decision_tree",
    "split": "test",
    "feature_set": best_feature_set,
    "train_f1": evaluate_model(y_train, model.predict(X_train))["f1"],
    "f1_gap": evaluate_model(y_train, model.predict(X_train))["f1"] - metrics["f1"],
})
results_feature_sets

## 3. Evaluación y guardado de resultados

Se muestra el informe de clasificación, se guardan las métricas, el modelo entrenado y la matriz de confusión para poder utilizarlos posteriormente en la comparativa final.

### Interpretacion esperada

El arbol puede lograr buenos resultados, pero tambien puede sobreajustar. Por eso se guarda `train_f1` y `f1_gap`: una diferencia grande entre entrenamiento y test indica que el modelo memoriza demasiado el conjunto de entrenamiento.


In [ ]:
print(get_classification_report(y_test, y_pred))
save_metrics(results_feature_sets.to_dict(orient="records"), RESULTS_DIR / "dt_feature_set_comparison.csv")
save_metrics(metrics, RESULTS_DIR / "dt_base_metrics.csv")
save_model(model, MODELS_DIR / "decision_tree_base.joblib")
save_confusion_matrix_plot(model, X_test, y_test, IMG_DIR / "decision_tree_base_cm.png")

## 4. Conclusiones

Este notebook permite obtener una primera aproximación con Decision Tree.  
El árbol de decisión es fácil de interpretar, pero puede tender al sobreajuste, por lo que se comparará posteriormente con modelos más robustos como Random Forest.